In [9]:
import xgboost as xgb
import pandas as pd
from sklearn.model_selection import train_test_split
from matplotlib import pyplot as plt
from sklearn.model_selection import GridSearchCV
from sklearn.metrics import make_scorer, ndcg_score
import numpy as np

# load data

In [10]:
# load training dataset
train_path = "data/basic_dataset.csv"
chunk = pd.read_csv(train_path, chunksize=1000000)
df = pd.concat(chunk)

# data preparation

In [11]:
# split data based on search ids
train_groups, val_groups = train_test_split(df[['srch_id']].drop_duplicates(), test_size=0.1, random_state=42)

# split the actual data on those groups
train_data = df[df['srch_id'].isin(train_groups['srch_id'])]
val_data = df[df['srch_id'].isin(val_groups['srch_id'])]

# prepare X and y for train/val set
X_train = train_data.drop(columns=['click_bool', 'booking_bool', 'relevance', 'position', 'gross_bookings_usd'])
y_train = train_data['relevance']

X_val = val_data.drop(columns=['click_bool', 'booking_bool', 'relevance', 'position', 'gross_bookings_usd'])
y_val = val_data['relevance']

In [12]:
# create groups (for DMatrix)
train_group = train_data.groupby('srch_id').size().to_list()
val_group = val_data.groupby('srch_id').size().to_list()

dtrain = xgb.DMatrix(X_train, label=y_train)
dtrain.set_group(train_group)

dval = xgb.DMatrix(X_val, label=y_val)
dval.set_group(val_group)

# Train model

In [17]:
params = {
    'objective': 'rank:pairwise',
    'eta': 0.1, # learning rate
    'gamma': 0.5, # minimum loss reduction required to further partition on a leaf node 
    'min_child_weight': 0.1, #  minimum sum of instance weights needed in a child
    'max_depth': 10, # max depth of the tree
    'eval_metric': 'ndcg' # 'map' for mean average precision; 'ndcg' for ndcg
}
param_grid = {
    'objective': ['rank:pairwise','rank:ndcg'],
    'eval_metric': ['ndcg'],
    'n_estimators': [50, 100, 150, 200], # epochs
    'eta': [0.01, 0.1, 0.2, 0.3], # learning rate
    'gamma': [0, 0.1, 0.2], # minimum loss reduction required to partition on a leaf
    'max_depth': [3, 5, 7, 9],  # max depth of the tree
    'subsample': [0.6, 0.8, 1.0],
    'colsample_bytree': [0.6, 0.8, 1.0],
    'lambda': [0, 1, 10],
    'alpha': [0, 1, 10]
}
param_grid = {
    'objective': ['rank:ndcg'],
    'eval_metric': ['ndcg'],
    'n_estimators': [100, 150, 200], # epochs
    'eta': [0.01, 0.1], # learning rate
    'gamma': [0, 0.1, 0.2], # minimum loss reduction required to partition on a leaf
    'max_depth': [3, 5, 7, 9],  # max depth of the tree
}

In [19]:
# scorer
def ndcg_scorer(y_true, y_pred, **kwargs):
    y_true_sorted = [y_true[np.argsort(-y_pred)]]
    return ndcg_score(y_true_sorted, y_true_sorted)

scorer = make_scorer(ndcg_scorer, needs_proba=True)

In [15]:
xgb_model = xgb.XGBRanker()

grid_search = GridSearchCV(estimator=xgb_model, param_grid=param_grid, scoring=scorer, cv=2, verbose=3, n_jobs=-1)
grid_search.fit(X_train, y_train, group=train_group)

print("Best parameters found: ", grid_search.best_params_)
print("Best NDCG score: ", -grid_search.best_score_)

Fitting 2 folds for each of 31104 candidates, totalling 62208 fits


KeyboardInterrupt: 

In [ ]:
# save best model to save_models/

# test best model
epochs = 200
evals_result = {} # save results into dict
bst = xgb.train(dtrain, epochs, params=grid_search.best_params_, evals=[(dtrain, 'train'), (dval, 'val')], early_stopping_rounds=10, evals_result=evals_result, verbose_eval=True)

best_iteration = bst.best_iteration
best_score = bst.best_score
bst.save_model(f'save_models/best_model_{best_iteration}_score_{best_score:.2f}.model')

In [ ]:
# plot training
train_evals = evals_result['train']['ndcg']
val_evals = evals_result['val']['ndcg']
plt.figure(figsize=(10, 5))
plt.plot(train_evals, label='Train NDCG')
plt.plot(val_evals, label='Validation NDCG')
plt.xlabel('Iterations')
plt.ylabel('NDCG')
plt.title('Training Vs. Validation NDCG')
plt.legend()
plt.ylim()
plt.show()

In [ ]:
# validation set
preds = bst.predict(dval)
print(preds)

Run model on the test_data:

In [ ]:
# load test set
test_path = "data/test_set_VU_DM.csv"
chunk = pd.read_csv(test_path, chunksize=1000000)
test_df = pd.concat(chunk)

# copied from data_prep.ipynb
test_df = test_df.drop(['date_time','orig_destination_distance','comp1_rate','comp1_inv','comp1_rate_percent_diff','comp2_rate','comp2_inv','comp2_rate_percent_diff','comp3_rate','comp3_inv','comp3_rate_percent_diff','comp4_rate','comp4_inv','comp4_rate_percent_diff','comp5_rate','comp5_inv','comp5_rate_percent_diff','comp6_rate','comp6_inv','comp6_rate_percent_diff','comp7_rate','comp7_inv','comp7_rate_percent_diff','comp8_rate','comp8_inv','comp8_rate_percent_diff','srch_query_affinity_score', 'visitor_hist_starrating','visitor_hist_adr_usd'],axis=1)

In [ ]:
dtest = xgb.DMatrix(test_df)

In [ ]:
test_preds = bst.predict(dtest)
print(test_preds)

In [ ]:
# add predictions as column
test_df['prediction'] = test_preds

grouped = test_df.groupby('srch_id').apply(lambda x: x.sort_values('prediction', ascending=False))

# Reset the index to flatten the DataFrame
grouped = grouped.reset_index(drop=True)

# Select the top 5 for each search_id
top5 = grouped.groupby('srch_id')

print(top5)

In [ ]:
grouped[['srch_id', 'prop_id']].to_csv('kaggle_predictions/xgboost_predictions.csv', index=False, sep=',')